# Audit 079: Circular Reference Sabotage
This notebook tests the cross-referencing logic to ensure that `CircularReferenceWarning` is thrown and cycles are broken when expanding text.

In [ ]:
import sys
import warnings

# Add path to CoChem-SCRIBE
sys.path.append('.')

from cochem_scribe.formatting.cross_referencing import ReferenceNode, RecursiveExpander, CircularReferenceWarning

def test_circular_reference():
    expander = RecursiveExpander()
    
    # 1. The Circular Reference Sabotage
    node_fig1 = ReferenceNode("fig_1", "Structure of complex, detailed in \\ref{table_1}")
    node_table1 = ReferenceNode("table_1", "Energetic data for the structure in \\ref{fig_1}")
    
    expander.add_node(node_fig1)
    expander.add_node(node_table1)
    
    # 3. Execution Guarantee
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        
        # 2. Physics Audit: Run expander
        result = expander.expand("fig_1")
        
        # Validation
        assert len(w) > 0, "No warnings were thrown!"
        warning_thrown = any(issubclass(warn.category, CircularReferenceWarning) for warn in w)
        assert warning_thrown, "CircularReferenceWarning was not thrown!"
        
        print(f"Warning successfully caught: {w[-1].message}")
        print(f"Cycle successfully broken, result: {result}")
        assert "\\ref{fig_1}" in result, "Cycle was not correctly broken with a terminal raw string."
        print("Audit 079 PASSED.")

test_circular_reference()
